Data Preprocessing Notebook

In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

Read training file and create dataframe

In [2]:
df = pd.read_csv('train.csv')

# Clean column names + ensure type is string
df.columns = df.columns.str.strip()
df["type"] = df["type"].astype(str)

# tolerance for float comparisons (important)
ATOL = 1e-6

# ADD THIS: use a train-derived threshold (more robust than hardcoding)
AMOUNT_THRESHOLD = df["amount"].quantile(0.999)


Check balances based on the type

In [3]:
# Add tolerant versions (more reliable than != for floats)
df["check_balanceOrg_tol"] = 0
df["check_balanceDest_tol"] = 0

# PAYMENT: origin should decrease by amount
mask = df["type"] == "PAYMENT"
exp = df.loc[mask, "oldbalanceOrg"] - df.loc[mask, "amount"]
df.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df.loc[mask, "newbalanceOrig"], exp, atol=ATOL)).astype(int)

# CASH_IN: origin should increase by amount
mask = df["type"] == "CASH_IN"
exp = df.loc[mask, "oldbalanceOrg"] + df.loc[mask, "amount"]
df.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df.loc[mask, "newbalanceOrig"], exp, atol=ATOL)).astype(int)

# CASH_OUT: origin should decrease by amount
mask = df["type"] == "CASH_OUT"
exp = df.loc[mask, "oldbalanceOrg"] - df.loc[mask, "amount"]
df.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df.loc[mask, "newbalanceOrig"], exp, atol=ATOL)).astype(int)

# TRANSFER: origin decreases, destination increases
mask = df["type"] == "TRANSFER"
exp_org = df.loc[mask, "oldbalanceOrg"] - df.loc[mask, "amount"]
exp_dst = df.loc[mask, "oldbalanceDest"] + df.loc[mask, "amount"]
df.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df.loc[mask, "newbalanceOrig"], exp_org, atol=ATOL)).astype(int)
df.loc[mask, "check_balanceDest_tol"] = (~np.isclose(df.loc[mask, "newbalanceDest"], exp_dst, atol=ATOL)).astype(int)

# DEBIT: origin decreases, destination increases
mask = df["type"] == "DEBIT"
exp_org = df.loc[mask, "oldbalanceOrg"] - df.loc[mask, "amount"]
exp_dst = df.loc[mask, "oldbalanceDest"] + df.loc[mask, "amount"]
df.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df.loc[mask, "newbalanceOrig"], exp_org, atol=ATOL)).astype(int)
df.loc[mask, "check_balanceDest_tol"] = (~np.isclose(df.loc[mask, "newbalanceDest"], exp_dst, atol=ATOL)).astype(int)


Check the amount of the transaction and flags large transactions

In [4]:
df["check_Amount_Size"] = (df["amount"] > AMOUNT_THRESHOLD).astype(int)


Extra features

In [5]:
df["deltaOrg"]  = df["oldbalanceOrg"] - df["newbalanceOrig"]
df["deltaDest"] = df["newbalanceDest"] - df["oldbalanceDest"]
df["dest_is_merchant"] = df["nameDest"].astype(str).str.startswith("M").astype(int)


Saving to new csv

In [6]:
# Load test and apply the SAME feature steps
df_test = pd.read_csv("test.csv")
df_test.columns = df_test.columns.str.strip()
df_test["type"] = df_test["type"].astype(str)

# Balance checks (tolerant)
df_test["check_balanceOrg_tol"] = 0
df_test["check_balanceDest_tol"] = 0

mask = df_test["type"] == "PAYMENT"
exp = df_test.loc[mask, "oldbalanceOrg"] - df_test.loc[mask, "amount"]
df_test.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df_test.loc[mask, "newbalanceOrig"], exp, atol=ATOL)).astype(int)

mask = df_test["type"] == "CASH_IN"
exp = df_test.loc[mask, "oldbalanceOrg"] + df_test.loc[mask, "amount"]
df_test.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df_test.loc[mask, "newbalanceOrig"], exp, atol=ATOL)).astype(int)

mask = df_test["type"] == "CASH_OUT"
exp = df_test.loc[mask, "oldbalanceOrg"] - df_test.loc[mask, "amount"]
df_test.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df_test.loc[mask, "newbalanceOrig"], exp, atol=ATOL)).astype(int)

mask = df_test["type"] == "TRANSFER"
exp_org = df_test.loc[mask, "oldbalanceOrg"] - df_test.loc[mask, "amount"]
exp_dst = df_test.loc[mask, "oldbalanceDest"] + df_test.loc[mask, "amount"]
df_test.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df_test.loc[mask, "newbalanceOrig"], exp_org, atol=ATOL)).astype(int)
df_test.loc[mask, "check_balanceDest_tol"] = (~np.isclose(df_test.loc[mask, "newbalanceDest"], exp_dst, atol=ATOL)).astype(int)

mask = df_test["type"] == "DEBIT"
exp_org = df_test.loc[mask, "oldbalanceOrg"] - df_test.loc[mask, "amount"]
exp_dst = df_test.loc[mask, "oldbalanceDest"] + df_test.loc[mask, "amount"]
df_test.loc[mask, "check_balanceOrg_tol"] = (~np.isclose(df_test.loc[mask, "newbalanceOrig"], exp_org, atol=ATOL)).astype(int)
df_test.loc[mask, "check_balanceDest_tol"] = (~np.isclose(df_test.loc[mask, "newbalanceDest"], exp_dst, atol=ATOL)).astype(int)

# Amount size (use SAME threshold from train)
df_test["check_Amount_Size"] = (df_test["amount"] > AMOUNT_THRESHOLD).astype(int)

# Extra features
df_test["deltaOrg"]  = df_test["oldbalanceOrg"] - df_test["newbalanceOrig"]
df_test["deltaDest"] = df_test["newbalanceDest"] - df_test["oldbalanceDest"]
df_test["dest_is_merchant"] = df_test["nameDest"].astype(str).str.startswith("M").astype(int)

# Save both
df.to_csv("train_with_new_features.csv", index=False)
df_test.to_csv("test_with_new_features.csv", index=False)
print("Saved train_with_new_features.csv and test_with_new_features.csv")


Saved train_with_new_features.csv and test_with_new_features.csv
